In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset.jsonl")

Generating train split: 10 examples [00:00, 830.67 examples/s]


In [10]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Conversation:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_example)

Map: 100%|██████████| 10/10 [00:00<00:00, 328.27 examples/s]


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "TinyLlama v1.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights: 100%|██████████| 201/201 [00:14<00:00, 13.91it/s]


In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # important for LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677
